# AgentFlow with Flow-GRPO

Train the planner for 10 sequential parameter updates on one A100 80GB. Each update uses 32 training questions and 8 sampled trajectories per question, with at most 3 interaction turns. Model weights and AdamW state carry forward between updates.

After training, evaluate the trained planner on 125 Bamboogle questions using seeds 42, 43 and 44. Each trial generates one answer per question, giving 375 answers in total. GPT-4o grades the saved answers locally after GPU inference returns.


## 1. Dependencies


In [1]:
%pip install -q modal==1.6.4 pandas==2.2.3 datasets==4.0.0 python-dotenv

Note: you may need to restart the kernel to use updated packages.


## 2. Configuration

`TRAINING_STEPS` counts parameter updates. `MAX_TURNS` limits training interaction turns; `EVAL_MAX_TURNS` limits evaluation interaction turns.

For a new run, choose an unused `RUN_NAME`. To continue an interrupted run, keep its name and settings and set `RESUME_TRAINING=True`; the cloud checkpoint contains the latest completed update and any completed evaluation trials. Set `RUN_TRAINING=False` to read local results and continue any unfinished grading.


In [2]:
from pathlib import Path
import os
import sys

MODAL_PROFILE = "neu-info5100-oak-spr-2025"
GPU = "H200"
RUN_NAME = "qwen7b-flow-grpo-10steps-h200-v2"
RUN_TRAINING = True
RESUME_TRAINING = True
RUN_JUDGE = True
TRAINING_STEPS = 10
MODEL_REVISION = "a09a35458c702b33eeacc393d103063234e8bc28"
SEED = 42
QUESTIONS_PER_STEP = 32
SEARCH_QUESTIONS = 16
ROLLOUTS_PER_QUESTION = 8
ROLLOUT_BATCH_SIZE = 32
TOOL_WORKERS = 32
REWARD_WORKERS = 8
UPDATE_BATCH_SIZE = 1
REUSE_IDENTICAL_REQUESTS = True
MAX_TURNS = 3
TEMPERATURE = 0.5
PLANNER_MAX_NEW_TOKENS = 2048
MODULE_MAX_NEW_TOKENS = 512
MAX_NEW_TOKENS = 512
LEARNING_RATE = 1e-6
KL_BETA = 0.001
CLIP_EPSILON = 0.2
MAX_GRAD_NORM = 1.0
LORA_RANK = 16
LORA_ALPHA = 32
LORA_TARGET_MODULES = ["q_proj", "k_proj", "v_proj", "o_proj",
                       "gate_proj", "up_proj", "down_proj"]
LOGPROB_TOKEN_CHUNK = 128
SHUFFLE_BUFFER = 1024
SEARCH_MODEL = "gpt-5.4-mini"
EMBEDDING_MODEL = "text-embedding-3-small"
CHUNK_WORDS = 200
CHUNK_OVERLAP_WORDS = 20
RETRIEVAL_TOP_K = 10
PAGE_MAX_WORDS = 1_000_000
JUDGE_MODEL = "gpt-4o-2024-08-06"
EVAL_QUESTION_LIMIT = 125
EVAL_SEEDS = [42, 43, 44]
EVAL_BATCH_SIZE = 32
EVAL_MAX_TURNS = 10
EVAL_TEMPERATURE = 0.7

os.environ["MODAL_PROFILE"] = MODAL_PROFILE
HERE = Path.cwd()
if str(HERE) not in sys.path:
    sys.path.insert(0, str(HERE))

import pandas as pd
from IPython.display import display, Markdown
from baseline.config import MODEL_ID
from baseline.results import read_json, write_json
from flow_grpo.data import training_batches
from baseline import evaluation as ev
from agent_flow import results

TRAIN_SETTINGS = dict(
    training_steps=TRAINING_STEPS,
    gpu=GPU, model_revision=MODEL_REVISION, seed=SEED,
    questions_per_step=QUESTIONS_PER_STEP, rollouts_per_question=ROLLOUTS_PER_QUESTION,
    rollout_batch_size=ROLLOUT_BATCH_SIZE, tool_workers=TOOL_WORKERS, max_turns=MAX_TURNS,
    reward_workers=REWARD_WORKERS, update_batch_size=UPDATE_BATCH_SIZE,
    reuse_identical_requests=REUSE_IDENTICAL_REQUESTS,
    temperature=TEMPERATURE, planner_max_new_tokens=PLANNER_MAX_NEW_TOKENS,
    module_max_new_tokens=MODULE_MAX_NEW_TOKENS, max_new_tokens=MAX_NEW_TOKENS,
    learning_rate=LEARNING_RATE, kl_beta=KL_BETA, clip_epsilon=CLIP_EPSILON,
    max_grad_norm=MAX_GRAD_NORM, lora_rank=LORA_RANK, lora_alpha=LORA_ALPHA,
    lora_target_modules=LORA_TARGET_MODULES, logprob_token_chunk=LOGPROB_TOKEN_CHUNK,
    shuffle_buffer=SHUFFLE_BUFFER,
    search_model=SEARCH_MODEL, embedding_model=EMBEDDING_MODEL,
    chunk_words=CHUNK_WORDS, chunk_overlap_words=CHUNK_OVERLAP_WORDS,
    retrieval_top_k=RETRIEVAL_TOP_K, page_max_words=PAGE_MAX_WORDS,
    judge_model=JUDGE_MODEL,
    data_sources=[
        dict(dataset="PeterJinGo/nq_hotpotqa_train",
             revision="b7d80abfee334a7a91cb377544f09180d58b34f6",
             answer_column="golden_answers", questions=SEARCH_QUESTIONS),
        dict(dataset="zwhe99/DeepMath-103K",
             revision="5cf055d1fe3d7a2eb19719ac020211469736ae44",
             answer_column="final_answer", questions=QUESTIONS_PER_STEP - SEARCH_QUESTIONS),
    ],
)
EVAL_SETTINGS = dict(
    question_limit=EVAL_QUESTION_LIMIT, seeds=EVAL_SEEDS, gpu=GPU,
    model_revision=MODEL_REVISION, batch_size=EVAL_BATCH_SIZE,
    max_turns=EVAL_MAX_TURNS, temperature=EVAL_TEMPERATURE,
    planner_max_new_tokens=PLANNER_MAX_NEW_TOKENS,
    module_max_new_tokens=MODULE_MAX_NEW_TOKENS, max_new_tokens=MAX_NEW_TOKENS,
    tool_workers=TOOL_WORKERS, reuse_identical_requests=REUSE_IDENTICAL_REQUESTS,
    search_model=SEARCH_MODEL, embedding_model=EMBEDDING_MODEL,
    chunk_words=CHUNK_WORDS, chunk_overlap_words=CHUNK_OVERLAP_WORDS,
    retrieval_top_k=RETRIEVAL_TOP_K, page_max_words=PAGE_MAX_WORDS,
    judge_model=JUDGE_MODEL,
)
OUTPUT_ROOT = HERE / "outputs"
RUN_DIR = OUTPUT_ROOT / RUN_NAME
if RUN_TRAINING and not RESUME_TRAINING and RUN_DIR.exists():
    raise FileExistsError("run name already used")
benchmark = read_json(HERE / "bamboogle.json")
questions = benchmark["rows"][:EVAL_QUESTION_LIMIT]
print("Local result directory:", RUN_DIR)


Local result directory: /mnt/c/Users/flywi/Desktop/gitRepos/neu-self-improve-ai/week_09/outputs/qwen7b-flow-grpo-10steps-h200-v2


## 3. Prepare Training Batches

Prepare 10 disjoint batches before allocating the GPU: 16 Search-R1 and 16 DeepMath questions per update, totaling 320 questions. Exclude exact normalized Bamboogle matches. A resumed run uses its checkpoint's saved batches.


In [3]:
batches = []
if RUN_TRAINING and not RESUME_TRAINING:
    batches = training_batches(TRAIN_SETTINGS, [row["question"] for row in benchmark["rows"]])
    display(pd.DataFrame([dict(step=i, source=row["source"])
                          for i, batch in enumerate(batches, 1) for row in batch])
            .groupby(["step", "source"]).size().rename("questions").unstack(fill_value=0))


## 4. Train and Evaluate

Provide `OPENAI_API_KEY` in `.env`. Training rewards are required for every update. After the last update, benchmark inference runs automatically with at most 10 interaction turns per question and planner temperature 0.7.

The latest checkpoint is saved after every update and every completed evaluation trial. Progress logs remain in Modal; the notebook waits for the final result. A failed, unfinished update is repeated when resuming. Training reward is not benchmark accuracy.


In [4]:
if RUN_TRAINING:
    import modal
    from dotenv import dotenv_values

    api_key = dotenv_values(HERE / ".env", encoding="utf-8-sig").get("OPENAI_API_KEY")
    if not api_key:
        raise ValueError("Set OPENAI_API_KEY in .env before training.")
    tool_secret = modal.Secret.from_dict({"OPENAI_API_KEY": api_key})
    del api_key
    if RESUME_TRAINING:
        manifest = read_json(RUN_DIR / "manifest.json")
        if (manifest["training_settings"] != TRAIN_SETTINGS
                or manifest["protocol"]["settings"] != EVAL_SETTINGS):
            raise ValueError("Resume with the saved training and evaluation settings.")
    else:
        RUN_DIR, manifest = results.prepare_run(OUTPUT_ROOT, RUN_NAME, EVAL_SETTINGS)
        manifest["protocol"]["method"] = "agentflow_flow_grpo"
        manifest["training_settings"] = TRAIN_SETTINGS
        write_json(RUN_DIR / "manifest.json", manifest)

    from flow_grpo.run_modal_app import create_app
    app, train_and_evaluate = create_app(GPU, tool_secret)
    print("Training and evaluation submitted. Waiting for results; progress stays in Modal.")
    with app.run():
        output = train_and_evaluate.remote(
            RUN_NAME, batches, TRAIN_SETTINGS, [q["question"] for q in questions],
            EVAL_SETTINGS, RESUME_TRAINING)
    training = output["training"]
    write_json(RUN_DIR / "training_result.json", training)
    saved = results.prepare_response({"output": output["evaluation"]}, manifest, questions)
    write_json(RUN_DIR / "sequential_response.json", saved)
    print("Saved training history and benchmark answers.")
else:
    manifest = read_json(RUN_DIR / "manifest.json")
    training = read_json(RUN_DIR / "training_result.json")
    saved = read_json(RUN_DIR / "sequential_response.json")
records = results.response_records(saved)


Training and evaluation submitted. Waiting for results; progress stays in Modal.
Saved training history and benchmark answers.


## 5. Training and Inference Results

Training times below cover completed updates, including trajectory collection, reward judging and parameter updates. They exclude initial model loading, checkpoint writes, failed attempts and final benchmark evaluation. Memory is peak PyTorch reserved GPU memory in GiB.


In [5]:
print("Actual GPU:", training["gpu"])
print("Completed training updates:", len(training["steps"]))
step_rows = [dict(
    step=step["step"], reward=step["update"]["mean_training_reward"],
    loss=step["update"]["loss"], gradient_norm=step["update"]["gradient_norm"],
    adapter_change_l2=step["update"]["adapter_change_l2"],
    seconds=step["server_total_seconds"], peak_gpu_reserved_gib=step["peak_gpu_reserved_gib"],
) for step in training["steps"]]
display(pd.DataFrame(step_rows))
print("Completed training update time (hours):", round(sum(r["seconds"] for r in step_rows) / 3600, 3))
display(pd.DataFrame([dict(step=step["step"], **stage)
                      for step in training["steps"] for stage in step["stages"]]).round(3))
display(pd.DataFrame([dict(trial=trial["trial"], seed=trial["seed"],
                          trial_seconds=trial["trial_seconds"],
                          peak_gpu_reserved_gib=trial["peak_gpu_reserved_gib"])
                      for trial in saved["output"]["outputs"]]).round(3))


Actual GPU: NVIDIA H200
Completed training updates: 10


,step,reward,loss,gradient_norm,adapter_change_l2,seconds,peak_gpu_reserved_gib
0,1,0.457031,1.775334e-09,0.060460,0.004625,1847.150907,62.375000
1,2,0.550781,1.043851e-06,0.068406,0.003302,2173.719835,59.750000
2,3,0.371094,1.254969e-06,0.066030,0.002695,1964.192894,66.871094
3,4,0.449219,1.570371e-06,0.070688,0.002447,2083.379607,63.994141
4,5,0.441406,1.200877e-06,0.050573,0.002169,2014.044410,76.013672
5,6,0.343750,1.010503e-06,0.077830,0.002021,2367.479840,77.222656
6,7,0.386719,1.122945e-06,0.080300,0.001910,1910.290434,60.804688
7,8,0.402344,1.041786e-06,0.055880,0.001803,2548.791846,68.609375
8,9,0.550781,1.289639e-06,0.073872,0.001737,1823.450678,59.474609
9,10,0.429688,1.169697e-06,0.053225,0.001643,2255.621814,57.210938


Completed training update time (hours): 5.83


,step,stage,seconds,peak_gpu_reserved_gib,qwen_generation_seconds,qwen_requests_reused
0,1,rollout,1533.999,62.375,1508.343,105.0
1,1,reward_judging,27.867,14.414,NaN,NaN
2,1,parameter_update,284.791,19.627,NaN,NaN
3,2,rollout,1840.858,59.750,1662.245,176.0
4,2,reward_judging,28.622,14.902,NaN,NaN
5,2,parameter_update,303.554,18.408,NaN,NaN
6,3,rollout,1637.968,66.871,1580.324,104.0
7,3,reward_judging,29.259,14.902,NaN,NaN
8,3,parameter_update,296.275,20.383,NaN,NaN
9,4,rollout,1755.656,63.994,1713.883,102.0


,trial,seed,trial_seconds,peak_gpu_reserved_gib
0,1,42,1513.029,90.666
1,2,43,1519.838,111.229
2,3,44,1480.590,102.285


## 6. Grade Saved Benchmark Answers

GPT-4o uses the same grading instructions as the baseline and untrained AgentFlow. Each decision is saved in the corresponding answer's `judge` field in `sequential_response.json`. Already graded answers are skipped.


In [6]:
if RUN_JUDGE:
    from dotenv import dotenv_values
    pending = [record for record in records if record.get("judge") is None]
    if pending:
        api_key = dotenv_values(HERE / ".env", encoding="utf-8-sig").get("OPENAI_API_KEY")
        judged = len(records) - len(pending)
        status = display(Markdown(f"**Grading:** {judged}/{len(records)}"), display_id=True)
        try:
            for record in pending:
                verdict = ev.judge_answer(
                    record["question"], record["reference_answers"], record["response"],
                    manifest["protocol"]["settings"]["judge_model"], api_key,
                    prompt=manifest["protocol"]["judge_prompt"])
                results.save_judgment(RUN_DIR, record["trial"], record["question_id"], verdict)
                judged += 1
                if judged % 25 == 0 or judged == len(records):
                    status.update(Markdown(f"**Grading:** {judged}/{len(records)}"))
        finally:
            del api_key
            records = results.load_records(RUN_DIR)
    else:
        print("All saved answers have already been graded.")

**Grading:** 375/375

## 7. Bamboogle Accuracy


In [7]:
report = ev.summary(records, manifest)
display(pd.DataFrame(report["trials"]))
if report["complete"]:
    print(f"Mean accuracy: {report['mean_accuracy_percent']:.2f}%")
    print(f"Sample standard deviation: {report['sample_std_percentage_points']:.2f} percentage points")
else:
    print("Benchmark grading is incomplete.")

,trial,seed,expected,generated,judged,correct,accuracy_percent,status
0,1,42,125,125,125,61,48.8,complete
1,2,43,125,125,125,57,45.6,complete
2,3,44,125,125,125,53,42.4,complete


Mean accuracy: 45.60%
Sample standard deviation: 3.20 percentage points


## Saved Files

Paths below are relative to the notebook folder.

| Path | Contents |
| --- | --- |
| `outputs/<RUN_NAME>/manifest.json` | Training/evaluation settings, model identity and prompts. |
| `outputs/<RUN_NAME>/training_result.json` | Completed training steps: questions, sampled answers, rewards, update metrics, stage times and memory peaks. |
| `outputs/<RUN_NAME>/sequential_response.json` | Three evaluation trials. Each contains its seed, timings, 125 questions with reference answers, interaction histories, generated answers and grading decisions. |

The Modal Volume `week09-flow-grpo-checkpoints` stores `<RUN_NAME>/checkpoint.pt`: the latest planner adapter, optimizer and random-generator states, training batches/history and completed evaluation trials. It is replaced after each completed update or trial. See `flow_grpo/README.md` for the implementation and resume procedure.
